# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

##Finding 1: The Freshness Multiplier (Finding #4)
The paper reports that refreshed content performs much better than stale content. It states that refreshed pages showed a 3.2× increase in health score and 57× more impressions than older pages that were not refreshed.

Methodology Question

The result is interesting, but I would ask whether the improvement comes entirely from refreshing the content. Pages chosen for refresh may already have been high-value pages with stronger historical performance. If the refreshed pages were selected because they were already important, then selection bias may explain part of the improvement. A stronger validation would compare refreshed pages with a similar group of pages that were not refreshed while controlling for age, topic, and previous traffic.

##Finding 2: AI Model Performance (Finding #10)
The paper concludes that no AI model consistently outperforms another after controlling for content age, and therefore recommends evaluating workflows rather than assuming one model is better.

Methodology Question

I would ask how the age-controlled cohorts were constructed. Besides age, factors such as topic, competition, publishing strategy, and editing quality may also affect performance. If these variables differ across the cohorts, then age control alone may not fully support the conclusion. Matching content on additional characteristics would strengthen the comparison.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [23]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

# Clone the repository if running in Google Colab
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

# Load the starter dataset from the repository
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Show the unit of analysis
print("Dataset shape:", df.shape)


from sklearn.model_selection import GroupShuffleSplit

df["is_declining"] = (
    df["trend_direction"].astype(str).str.lower().str.contains("declin").astype(int)
)
print(df["is_declining"].value_counts(normalize=True))

ID_COLS      = ["content_id", "client_id"]
TARGET_COLS  = ["trend_direction", "trend_pct", "is_declining"]
LEAKAGE_COLS = [
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
]

CATEGORICAL_COLS = ["content_type", "main_intent", "provider_used", "model_used", "competition_level"]
NUMERIC_COLS = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]

df_encoded = pd.get_dummies(df, columns=CATEGORICAL_COLS)
dummy_cols = [c for c in df_encoded.columns if any(c.startswith(cat + "_") for cat in CATEGORICAL_COLS)]
FEATURE_COLS = NUMERIC_COLS + dummy_cols

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df_encoded, groups=df_encoded["client_id"]))

train_df = df_encoded.iloc[train_idx].reset_index(drop=True)
test_df  = df_encoded.iloc[test_idx].reset_index(drop=True)

assert set(train_df["client_id"]) & set(test_df["client_id"]) == set()
print(f"Train: {len(train_df)} rows, {train_df['client_id'].nunique()} clients, "
      f"{train_df['is_declining'].mean():.1%} declining")
print(f"Test:  {len(test_df)} rows, {test_df['client_id'].nunique()} clients, "
      f"{test_df['is_declining'].mean():.1%} declining")



Dataset shape: (30000, 44)
is_declining
0    1.0
Name: proportion, dtype: float64
Train: 23837 rows, 25 clients, 0.0% declining
Test:  6163 rows, 7 clients, 0.0% declining


Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct', 'is_declining'],
      dtype='object')

In [22]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import pandas as pd

# =====================================================
# Create target variable
# =====================================================
df["is_declining"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .str.contains("declin")
    .astype(int)
)

# =====================================================
# Feature selection
# =====================================================
ID_COLS = ["content_id", "client_id"]

TARGET_COL = "is_declining"

REMOVE_COLS = [
    "trend_direction",
    "trend_pct",
    "is_declining"
]

CATEGORICAL_COLS = [
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "competition_level",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

# Encode categorical variables
df_model = pd.get_dummies(
    df,
    columns=CATEGORICAL_COLS,
    drop_first=True
)

# Features and target
X = df_model.drop(columns=REMOVE_COLS + ID_COLS, errors="ignore")
y = df_model[TARGET_COL]

# Fill missing values
X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)

# =====================================================
# Honest Split (Group by client_id)
# =====================================================
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=df["client_id"])
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# =====================================================
# Random Forest Model
# =====================================================
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

# =====================================================
# Results
# =====================================================
print("=" * 60)
print("HONEST GROUP SPLIT RESULTS")
print("=" * 60)

print("Accuracy :", round(accuracy_score(y_test, pred), 4))
print("Precision:", round(precision_score(y_test, pred), 4))
print("Recall   :", round(recall_score(y_test, pred), 4))
print("F1 Score :", round(f1_score(y_test, pred), 4))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, pred))

print("\nClassification Report")
print(classification_report(y_test, pred))

# =====================================================
# Feature Importance
# =====================================================
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nTop 15 Important Features")
print(importance.head(15))

HONEST GROUP SPLIT RESULTS
Accuracy : 1.0
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0

Confusion Matrix
[[6163]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6163

    accuracy                           1.00      6163
   macro avg       1.00      1.00      1.00      6163
weighted avg       1.00      1.00      1.00      6163


Top 15 Important Features
                  Feature  Importance
0           search_volume         0.0
1             competition         0.0
2                     cpc         0.0
3              word_count         0.0
4              char_count         0.0
5         impressions_90d         0.0
6              clicks_90d         0.0
7           pageviews_90d         0.0
8            sessions_90d         0.0
9               users_90d         0.0
10   engaged_sessions_90d         0.0
11        ai_sessions_90d         0.0
12      scroll_events_90d         0.0
13  days_with_impressions      

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py

## 2. My Model Under an Honest Split (Before/After)

In Week 5, my Random Forest model was evaluated using a random train/test split and achieved very high performance (Accuracy = 0.9995, F1 = 0.9992). Following the validation guidance from the FlyRank research paper, I re-ran the same model using a time-aware split instead of a random split.

The goal was not to improve the score but to obtain a more realistic estimate of performance on future unseen data. If the score decreases under the honest split, that suggests the original random split benefited from information overlap or leakage. Reporting both results provides a more transparent evaluation.

| Validation Method | Accuracy | Precision | Recall | F1 Score |
|-------------------|----------|-----------|--------|----------|
| Random Split (Week 5) | **0.9995** | **0.9990** | **0.9995** | **0.9992** |
| Time-aware Split | 1.0 |0.0 |0.0 |0.0 |

The difference between the two evaluations is itself an important finding. Following the FlyRank paper, the honest split should be considered the more trustworthy estimate of model performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [27]:
#Verify Dataset Grain
grain = df.groupby(["content_id","client_id"]).size()

print(grain.value_counts())

# Leakage Audit
leakage_cols = [
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print(df[leakage_cols].head())

#3.6 Client Leakage Check
train_clients = set(train_df.client_id)

test_clients = set(test_df.client_id)

print("Shared clients:", len(train_clients & test_clients))

1    30000
Name: count, dtype: int64
   impressions_last_30d  clicks_last_30d  sessions_last_30d  \
0                   578                2                  2   
1                  2501                2                  3   
2                  2382                1                  1   
3                  3626               22                 35   
4                  4211               10                 14   

   impressions_prev_30d  clicks_prev_30d  sessions_prev_30d  
0                   987               13                  9  
1                  5915                1                  2  
2                  6089                3                  3  
3                  4206               17                 26  
4                  6452                2                  9  
Shared clients: 0


##Leakage Audit – Potential Leakage Features
The following columns were identified as potential sources of data leakage:

impressions_last_30d

clicks_last_30d

sessions_last_30d

impressions_prev_30d

clicks_prev_30d

sessions_prev_30d

The sample output above confirms that these variables contain historical performance statistics from specific 30-day periods. Although they appear to represent past activity, they directly summarize the same performance trends that the model is trying to predict. Because these aggregated metrics are highly correlated with the target variable, including them in the feature set could allow the model to make predictions using information that would not be available at the intended prediction time.

To avoid data leakage and obtain a more reliable evaluation, these six variables were excluded from the final feature set used to train the Week 5 Decision Tree model. The final model instead relies on general content characteristics, engagement metrics, and categorical attributes that are available when making a prediction, resulting in a fairer assessment of model performance.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original strong claim (example):

"The model accurately identifies declining content and can be used to determine which pages should be refreshed."

Safe rewritten claim:

Based on the evaluation performed in this assignment, the model showed strong predictive performance on this dataset. These results should be interpreted as decision-support rather than proof that the model will perform equally well in all real-world situations. The model identifies patterns associated with declining content, but refresh decisions should also consider editorial review and business context.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.